# Guided Search: Success and Peak Memory

One row per planned start/goal pair, per run. A run is a `guided_search.py`
tree search: it samples guides from the start term, attempts each one, and
expands the failures until its budget runs out: `--max-attempts`,
`--max-pair-time`, or both, in which case whichever runs out first.

Memory statistics compare each guided success against the brute-force cost of
proving the same pair, taken from the `problems.json` the run was built on.

Memory is reported as **peak RSS**

## TODO: Error bars since we now pool seeds!

In [ ]:
import altair as alt

import helpers as H
import plots as P

alt.theme.register("analysis", enable=True)(lambda: P.THEME)
alt.data_transformers.enable("default", max_rows=None)

In [ ]:
# Fix the language to inspect (loads `data/guided_search/<LANGUAGE>` and filters on it).
LANGUAGE = "math"  # "math", "prop"

# Empty takes every finished run folder under `data/guided_search/<LANGUAGE>`.
# Use name substrings such as ("12", "13") to select individual experiments,
# and `folder=` to read another folder under `data`, e.g. "guided_search.bak".
# `WHERE` keeps runs whose config.json has every value; a tuple allows any of its entries.
WHERE = {
    # "search_policy": "bfs",  # "bfs", "dfs"
    # "frontier": True,
    # "tree_width": (30, 15, 6),  # 2, 5, 6, 10, 15, 30
    # "sample_policy": "count",  # "count", "uniform"
    # "sampling_backoff": "gallop",  # "gallop", "linear"
    # "guides_per_attempt": (1, 5),  # 1, 2, 5
}
RUNS, INCOMPLETE, df, meta = H.load_comparisons((), LANGUAGE, where=WHERE)
if INCOMPLETE:
    print("Incomplete:\n" + "\n".join(INCOMPLETE))

## Success

In [ ]:
H.success_summary(df)

In [ ]:
P.success_outcomes(H.outcome_counts(df), meta)

### How the search ended

One category per planned pair, successes included. A missing guide menu takes
precedence; otherwise it is the search's own stop reason: reached the goal, ran
out of a budget, or ran out of nodes to attempt.

In [ ]:
P.search_outcomes(H.search_outcomes(df), meta)

### Failed attempts

One row per attempt that did not reach the goal, by its stop reason. Split by
whether the attempt's pair was eventually proved by a later attempt or never.

In [ ]:
P.attempt_failures(H.attempt_failures(RUNS), meta)

## Peak memory

Left: guided workflow peak RSS (max over guide replay, sample construction, and
every attempt the pair ran) against the brute-force baseline on proved pairs.
Right: all pairs bucketed by brute-force proof cost, split by whether guided
search proved them. Every guided process runs under `--max-rss`, below the
`min_rss` the problems were selected with, so a guided success always peaks
below its brute-force proof.

In [ ]:
P.peak_memory(H.peak_memory(df), meta)

## Search effort

Attempts, search-tree depth, and wall time (summed over expansions and attempts),
in power-of-two buckets. `first success` counts successful pairs at the point they
first reached the goal; `whole search` counts every pair that ran an attempt by
its total attempts, deepest attempt, or total wall time.

In [ ]:
P.search_effort(H.search_effort(df), meta)

## Sampling overhead

Left: how many `sample` processes each draw a pair paid for started (past 1 try
means a try was killed at the RSS cap and stepped down by `sampling_backoff`).
Right: the share of each run's `sample` wall time spent in those killed tries.

In [ ]:
P.sampling_overhead(H.sampling_overhead(RUNS), meta)

## Pairwise solved

Left: pairs the row run proves guided and the column run does not. Right:
pairs proved guided by either run.

In [ ]:
P.pairwise_solved(H.pairwise_solved_diff(df), meta)

## Root e-graph density

Pairs grouped into equal-sized quantiles by the density (`nodes / classes`) of
the root e-graph (`depth == 0`). Left: guided success rate for `whole` vs
`frontier` across density quantiles. Right: paired `whole` vs `frontier`
outcomes in each bucket, showing both the regime shift from sparse to dense
e-graphs and their complementarity.

In [ ]:
P.egraph_density(H.egraph_density(RUNS, df), meta)